# Assignment 2 — Transformer Architecture Comparison (MNIST)

This notebook compares three transformer-style architectures adapted to MNIST (decoder-only, encoder-only, encoder-decoder).

## 1 — Setup

In [ ]:
import os, time, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 2 — Dataset (MNIST) and preprocessing

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_ds = torchvision.datasets.MNIST(root='/mnt/data', train=True, download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST(root='/mnt/data', train=False, download=True, transform=transform)
len(train_ds), len(test_ds)


In [ ]:
def img_to_tokens(img_tensor):
    arr = (img_tensor.squeeze(0).numpy() * 255.0).round().astype(np.int64)
    return arr.flatten().tolist()

def tokens_to_img(tokens):
    import numpy as _np
    arr = _np.array(tokens, dtype=_np.int64).reshape(28,28)
    arr = (arr.clip(0,255)/255.0).astype(_np.float32)
    return arr


## 3 — Shared hyperparameters and helpers

In [ ]:
SEQ_LEN = 28*28
VOCAB_SIZE = 256
EMBED_DIM = 128
BATCH_SIZE = 128
EPOCHS = 2
LR = 1e-3

def collate_tokens(batch):
    tokens = [img_to_tokens(x[0]) for x in batch]
    labels = [x[1] for x in batch]
    import torch as _torch
    return _torch.tensor(tokens, dtype=_torch.long), _torch.tensor(labels, dtype=_torch.long)

from torch.utils.data import DataLoader
def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_tokens)


## 4 — Model A: Decoder-only (GPT-style)

In [ ]:
import torch.nn.functional as F
class TinyGPT(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, emb=EMBED_DIM, nhead=4, nlayers=3, seq_len=SEQ_LEN):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb)
        self.pos_emb = nn.Embedding(seq_len, emb)
        layer = nn.TransformerDecoderLayer(d_model=emb, nhead=nhead, dim_feedforward=emb*4)
        self.transformer = nn.TransformerDecoder(layer, num_layers=nlayers)
        self.fc = nn.Linear(emb, vocab_size)
        self.seq_len = seq_len
    def forward(self, x):
        B,L = x.shape
        pos = torch.arange(0,L, device=x.device).unsqueeze(0).expand(B,-1)
        tok = self.token_emb(x) + self.pos_emb(pos)
        tgt_mask = torch.triu(torch.ones(L, L, device=x.device) * float('-inf'), diagonal=1)
        out = self.transformer(tgt=tok.permute(1,0,2), memory=tok.permute(1,0,2), tgt_mask=tgt_mask)
        out = out.permute(1,0,2)
        return self.fc(out)

def gpt_loss(logits, targets):
    return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))


### Train (demo, short)

In [ ]:
train_loader = make_loader(train_ds, batch_size=BATCH_SIZE)
gpt = TinyGPT().to(device)
opt = optim.Adam(gpt.parameters(), lr=LR)
gpt.train()
for epoch in range(1, EPOCHS+1):
    for xb, yb in train_loader:
        xb = xb.to(device)
        opt.zero_grad()
        logits = gpt(xb)
        loss = gpt_loss(logits, xb)
        loss.backward()
        opt.step()
        print('GPT demo step loss', loss.item())
        break


## 5 — Model B: Encoder-only (BERT-style)

In [ ]:
class TinyBERT(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, emb=EMBED_DIM, nhead=4, nlayers=3, seq_len=SEQ_LEN):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb)
        self.pos_emb = nn.Embedding(seq_len, emb)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb, nhead=nhead, dim_feedforward=emb*4)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.mlm_head = nn.Linear(emb, vocab_size)
        self.class_head = nn.Linear(emb, 10)
    def forward(self, x, mode='mlm'):
        B,L = x.shape
        pos = torch.arange(0,L, device=x.device).unsqueeze(0).expand(B,-1)
        h = self.token_emb(x) + self.pos_emb(pos)
        h = self.transformer(h.permute(1,0,2)).permute(1,0,2)
        if mode=='mlm':
            return self.mlm_head(h)
        elif mode=='class':
            return self.class_head(h.mean(dim=1))


In [ ]:
def mask_tokens(inputs, mask_token=0, mlm_prob=0.15):
    import numpy as _np
    if isinstance(inputs, torch.Tensor):
        arr = inputs.clone().cpu().numpy()
    else:
        arr = inputs.copy()
    B,L = arr.shape
    labels = -100 * _np.ones_like(arr, dtype=_np.int64)
    for i in range(B):
        mask_pos = _np.random.rand(L) < mlm_prob
        labels[i, mask_pos] = arr[i, mask_pos]
        arr[i, mask_pos] = mask_token
    import torch as _torch
    return _torch.tensor(arr, dtype=_torch.long), _torch.tensor(labels, dtype=_torch.long)

bert = TinyBERT().to(device)
opt = optim.Adam(bert.parameters(), lr=LR)
bert.train()
loader = make_loader(train_ds, batch_size=BATCH_SIZE)
for xb, yb in loader:
    masked_inp, mlm_labels = mask_tokens(xb)
    masked_inp = masked_inp.to(device); mlm_labels = mlm_labels.to(device)
    opt.zero_grad()
    logits = bert(masked_inp, mode='mlm')
    loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), mlm_labels.view(-1), ignore_index=-100)
    loss.backward(); opt.step()
    print('BERT demo step loss', loss.item())
    break


## 6 — Model C: Encoder-Decoder (T5-style)

In [ ]:
class TinyT5(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, emb=EMBED_DIM, nhead=4, nlayers=2, seq_len=SEQ_LEN):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb)
        self.pos_enc = nn.Embedding(seq_len, emb)
        self.pos_dec = nn.Embedding(seq_len, emb)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb, nhead=nhead, dim_feedforward=emb*4)
        dec_layer = nn.TransformerDecoderLayer(d_model=emb, nhead=nhead, dim_feedforward=emb*4)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=nlayers)
        self.fc = nn.Linear(emb, vocab_size)
    def forward(self, src, tgt):
        B,L = src.shape
        pos = torch.arange(0,L, device=src.device).unsqueeze(0).expand(B,-1)
        h = self.token_emb(src) + self.pos_enc(pos)
        mem = self.encoder(h.permute(1,0,2))
        tpos = torch.arange(0,tgt.shape[1], device=tgt.device).unsqueeze(0).expand(B,-1)
        t_emb = self.token_emb(tgt) + self.pos_dec(tpos)
        tgt_mask = torch.triu(torch.ones(tgt.shape[1], tgt.shape[1], device=src.device)*float('-inf'), diagonal=1)
        out = self.decoder(t_emb.permute(1,0,2), mem, tgt_mask=tgt_mask).permute(1,0,2)
        return self.fc(out)


In [ ]:
t5 = TinyT5().to(device)
opt = optim.Adam(t5.parameters(), lr=LR)
loader = make_loader(train_ds, batch_size=BATCH_SIZE)
for xb, yb in loader:
    tokens = xb.to(device)
    k = tokens.shape[1]//4
    src = torch.zeros_like(tokens); src[:, :k] = tokens[:, :k]
    tgt = tokens
    opt.zero_grad()
    logits = t5(src, tgt)
    loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), tgt.view(-1))
    loss.backward(); opt.step()
    print('T5 demo step loss', loss.item())
    break


## 7 — Save sample outputs and notebook

In [ ]:
import matplotlib.pyplot as plt
def sample_gpt(model):
    model.eval()
    tokens = torch.zeros((1, SEQ_LEN), dtype=torch.long, device=device)
    for i in range(SEQ_LEN):
        with torch.no_grad():
            logits = model(tokens[:, :i+1])[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            tokens[0,i] = torch.multinomial(probs, num_samples=1)[0,0]
    return tokens[0].cpu().numpy()

gpt_sample = sample_gpt(TinyGPT().to(device))
plt.imsave('/mnt/data/gpt_demo.png', tokens_to_img(gpt_sample), cmap='gray')

bert_small = TinyBERT().to(device)
xb, yb = collate_tokens([test_ds[i] for i in range(8)])
masked_inp, mlm_labels = mask_tokens(xb.numpy(), mask_token=0, mlm_prob=0.2)
with torch.no_grad():
    logits = bert_small(torch.tensor(masked_inp, dtype=torch.long).to(device), mode='mlm')
pred = logits.argmax(dim=-1).cpu().numpy()[0]
plt.imsave('/mnt/data/bert_demo.png', tokens_to_img(pred), cmap='gray')

t5_small = TinyT5().to(device)
tokens, _ = collate_tokens([test_ds[0]])
k = tokens.shape[1]//4
src = torch.zeros_like(tokens); src[:, :k] = tokens[:, :k]
with torch.no_grad():
    logits = t5_small(src.to(device), tokens.to(device))
pred = logits.argmax(dim=-1).cpu().numpy()[0]
plt.imsave('/mnt/data/t5_demo.png', tokens_to_img(pred), cmap='gray')

print('Saved demo images to /mnt/data/*.png')
